# Session 3 plan: evaluation only (no training, ~20-40 min)
Session 2 trained `patch / both heads / cond_prev=False / QAT` (`patch_both_prev0_K0_qat5_v2.pt`). Its report showed Kalman fusion
RMSE 0.102 (R² 0.673) against 0.120 for the absolute head alone. Two questions must be answered before that number can be reported:
1. **Is the gain from the delta head, or just from temporal smoothing?** New estimator `abs_smoothed` = same filter with zero delta.
2. **Does it survive without the true starting angle?** Session 2 seeded every run with the glove reading at t=0. `init='abs'` seeds with the model's first prediction instead (deployable).

Also fixes: the old Kalman sweep tuned the *rest-anchored* filter (wrong estimator); horizon bins now have stable labels.

**Before running:** re-upload `emg_common.py` (adds `init=`, `abs_smoothed`, `kalman_sweep`). Attach: session-1 dataset (`db2_processed/`) and the session-2 output saved as a dataset (contains `outputs/patch_both_prev0_K0_qat5_v2.pt`).
Sections 01, 02 (training), 03 and 05 are commented out. Click **Save Version** at the end.

# 01 · Preprocess and split (leakage-free)
Builds per-subject windows from NinaPro DB2 with:
* **repetition-based split** (train reps 1,3,4,6 / test reps 2,5; windows straddling a boundary are dropped),
* **causal** Butterworth band-pass + rectification (what the firmware can actually compute),
* normalisation constants computed from **training repetitions only** (per-channel EMG scale, glove min/max) and saved for the firmware,
* targets: absolute angle at the last sample, previous-window angle, **per-stride delta**, window delta, restimulus, repetition, segment and time index (for ordered evaluation).

Outputs: `/kaggle/working/db2_processed/<subject>/data.npz` + `norm_stats.json`. Save this notebook's output as a dataset and attach it to the other notebooks.

In [ ]:
# # --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
# import os, sys, glob, json
# for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
#     if os.path.exists(os.path.join(p, 'emg_common.py')):
#         sys.path.insert(0, os.path.abspath(p)); break
# else:
#     raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import os, sys, glob, json
sys.path.insert(0, '/kaggle/usr/lib/notebooks/quddusikashaf/emg_common')

In [ ]:
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# [session 2] skipped: processed data is attached as a dataset
# cfg = ec.Config(base_path=BASE, processed_dir=f'{WORK}/db2_processed', n_subjects=int(os.environ.get('EMG_N_SUBJECTS', 10)),
#                 seq_len=200, stride=50, causal_filter=True, train_reps=(1, 3, 4, 6), test_reps=(2, 5))
# subjects = ec.list_subjects(cfg)
# print('subjects:', subjects)
# print('window', cfg.seq_len, 'samples =', 1000 * cfg.seq_len / cfg.fs, 'ms | stride', cfg.stride, '=', 1000 * cfg.stride / cfg.fs, 'ms')

In [ ]:
# [session 2] skipped: processed data is attached as a dataset
# npz_files = [ec.process_subject(cfg, s) for s in subjects]

In [ ]:
# [session 2] skipped: processed data is attached as a dataset
# ---- summary and sanity checks ----
# rows = []
# for f in npz_files:
#     d = np.load(f)
#     tr, te = d['split'] == 0, d['split'] == 1
#     assert not (set(np.unique(d['rep'][tr])) & set(np.unique(d['rep'][te]))), 'train/test repetitions overlap'
#     rows.append({'subject': os.path.basename(os.path.dirname(f)), 'windows': len(d['split']), 'train': int(tr.sum()), 'test': int(te.sum()),
#                  'rest_frac_train': float((d['restim'][tr] == 0).mean()), 'train_reps': sorted(np.unique(d['rep'][tr]).tolist()),
#                  'test_reps': sorted(np.unique(d['rep'][te]).tolist()), 'emg_mean': float(d['sEMG'].mean()), 'emg_max': float(d['sEMG'].max())})
# summary = pd.DataFrame(rows); display(summary)
# delta_scale = ec.delta_scale_from_train(npz_files); rest_pose = ec.rest_pose_from_train(npz_files); thr = ec.rest_energy_threshold(npz_files)
# print('per-joint delta std (train):', np.round(delta_scale, 4))
# print('rest pose (train):', np.round(rest_pose, 3), '| rest energy threshold:', round(thr, 4))
# print('smallest delta std vs ap_fixed<10,3> LSB (1/128=0.0078):', round(float(delta_scale.min()), 4), '-> deltas MUST be standardised before quantisation')
# json.dump({'summary': rows, 'delta_scale': delta_scale.tolist(), 'rest_pose': rest_pose.tolist(), 'rest_energy_threshold': thr, 'cfg': cfg.to_dict()},
#           open(f'{cfg.processed_dir}/dataset_summary.json', 'w'), indent=2)

In [ ]:
# [session 2] skipped: processed data is attached as a dataset
# ---- visual check: one window and the ordered glove trace of a test run ----
# d = np.load(npz_files[0]); runs = ec.ordered_runs(npz_files[0], 'test'); r = max(runs, key=lambda x: len(x['angles']))
# fig, ax = plt.subplots(1, 3, figsize=(18, 4))
# ax[0].plot(d['sEMG'][100]); ax[0].set_title('one window: mu-law rectified EMG (12 ch)')
# ax[1].plot(r['angles'][:400]); ax[1].set_title(f'ordered test run: normalised angles ({len(r["angles"])} windows)')
# ax[2].plot(r['delta_stride'][:400]); ax[2].set_title('per-stride deltas (integration targets)')
# plt.tight_layout(); plt.savefig(f'{cfg.processed_dir}/preview.png', dpi=120); plt.show()

### Notes for the firmware
`norm_stats.json` per subject holds `emg_scale` (per channel), `glove_min`, `glove_max`, `mu`. The export notebook additionally writes the causal SOS filter
coefficients and `DELTA_SCALE` to `normalization_constants.h`. Nothing here uses future samples, so the PS can reproduce the pipeline sample by sample.

# 02 · Train RoFormer variants (absolute + delta heads, previous-angle conditioning, QAT)
One configurable experiment. Variants (`cfg.variant`):
* `roformer` – current design (k3/s1 conv, 200 tokens, LayerNorm, post-norm)
* `patch` – strided patch embedding (`patch_stride=8` → 25 tokens, ~8× fewer MACs)
* `noattn` – patch embedding + FFN only (attention ablation)
* `causal` – patch + causal attention + RMSNorm + last-token pooling (streaming / KV-cache friendly)

Heads: `absolute`, `delta`, `both`. `cond_prev=True` feeds the previous angle estimate into the head.
`unroll_steps=K>1` trains with **integration in the loop** (K consecutive windows, own estimate fed back with noise).
`qat_epochs>0` fine-tunes with fake quantisation matching `ap_fixed<W,I>` (`act_fmt`, `weight_fmt`).
W&B key is read from Kaggle Secrets (`wandb_key`), never hard-coded.

In [ ]:
# # --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
# import os, sys, glob, json
# for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
#     if os.path.exists(os.path.join(p, 'emg_common.py')):
#         sys.path.insert(0, os.path.abspath(p)); break
# else:
#     raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
# import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
# import emg_common as ec
# pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
# WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
# BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
# EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
# print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: attached dataset from session 1 (or same-session output of 01)
candidates = [f'{WORK}/db2_processed',
              '/kaggle/input/datasets/quddusikashaf/checkpoint-roformer/db2_processed',
              '/kaggle/input/checkpoint-roformer/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: attach the session-1 output dataset'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
# [session 3] skipped: model already trained in session 2 (loaded from the attached dataset in section 04)
# ---- SESSION 2: patch embedding, both heads, NO previous-angle conditioning, normalised loss, QAT with resid_t ----
# requires the re-uploaded emg_common.py (loss_norm / select_metric / resid_fmt)
# assert hasattr(ec.Config(), 'resid_fmt') and hasattr(ec.Config(), 'select_metric'), 're-upload the updated emg_common.py utility script'
# cfg = ec.Config(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{WORK}/outputs',
#                 variant='patch', patch_stride=8, heads='both', cond_prev=False, unroll_steps=0,
#                 d_model=128, d_ff=256, num_heads=8, num_layers=1, norm='layernorm', dropout=0.1,
#                 epochs=40, patience=8, lr=3e-4, batch_size=64,
#                 qat_epochs=5, act_fmt=(10, 3), resid_fmt=(16, 8), weight_fmt=(10, 3),
#                 loss_norm=True, select_metric='val_abs', delta_detach=False,
#                 use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_roformer')
# cfg.apply_variant()
# if EPOCHS_OVERRIDE is not None: cfg.epochs = cfg.qat_epochs = EPOCHS_OVERRIDE
# cfg.run_name = f'{cfg.variant}_{cfg.heads}_prev{int(cfg.cond_prev)}_K{cfg.unroll_steps}_qat{cfg.qat_epochs}_v2'
# print(cfg.run_name, '| tokens:', cfg.num_tokens, '| MACs:', ec.estimate_macs(cfg))
# run = ec.init_wandb(cfg)

In [ ]:
# [session 3] skipped: model already trained in session 2 (loaded from the attached dataset in section 04)
# model, info = ec.train_model(cfg, npz_files, run=run, tag=cfg.run_name)
# hist = pd.DataFrame(info['history']); display(hist.tail())
# hist.plot(x='epoch', y=['train_loss', 'val_loss'], logy=True, title=cfg.run_name); plt.show()

In [ ]:
# [session 3] skipped: model already trained in session 2 (loaded from the attached dataset in section 04)
# ---- window-level metrics (shuffled, independent windows) ----
# device = ec.get_device()
# eval_cfg = ec.Config.from_dict({**cfg.to_dict(), 'unroll_steps': 0})
# _, val_loader = ec.make_loaders(eval_cfg, npz_files)
# res = ec.predict_windows(model, val_loader, cfg, device); wm = ec.window_metrics(res); print('window metrics:', wm)
# ---- sequential metrics (ordered test runs, model fed its OWN estimate) ----
# rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)
# report = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr, feedback=True)
# seq = pd.DataFrame({k: {m: v[m] for m in ('rmse', 'r2', 'mae')} for k, v in report.items()}).T.sort_values('rmse'); display(seq)
# ec.wandb_log(run, {**{f'window/{k}': v for k, v in wm.items()}, **{f'seq/{k}_rmse': v['rmse'] for k, v in report.items()}})
# json.dump({'cfg': cfg.to_dict(), 'window': wm, 'sequential': report, 'checkpoint': info['checkpoint']},
#           open(f'{cfg.save_dir}/{cfg.run_name}_report.json', 'w'), indent=2)

In [ ]:
# [session 3] skipped: model already trained in session 2 (loaded from the attached dataset in section 04)
# ---- tracking plot on the longest test run ----
# runs = ec.ordered_runs(npz_files[0], 'test'); r = max(runs, key=lambda x: len(x['angles']))
# p = ec.predict_run(model, r, cfg, device, feedback=True)
# fig, axes = plt.subplots(5, 2, figsize=(18, 16)); n = min(600, len(r['angles']))
# for j, ax in enumerate(axes.ravel()):
#     ax.plot(r['angles'][:n, j], 'k', lw=1.5, label='truth')
#     if p['abs'] is not None: ax.plot(p['abs'][:n, j], 'r', alpha=.5, label='absolute head')
#     if p['delta'] is not None: ax.plot(ec.integrate(p['delta'], p['start'])[:n, j], 'b', alpha=.5, label='integrated delta')
#     if p['abs'] is not None and p['delta'] is not None:
#         ax.plot(ec.kalman_fuse(p['delta'], p['abs'], p['start'], energy=p['energy'], rest_threshold=thr, rest_pose=rest_pose)[:n, j], 'g', lw=1.5, label='kalman + rest anchor')
#     ax.set_title(ec.JOINT_NAMES[j]); ax.grid(alpha=.3)
# axes[0, 0].legend(); plt.tight_layout(); plt.savefig(f'{cfg.save_dir}/{cfg.run_name}_tracking.png', dpi=120); plt.show()
# if run is not None:
#     import wandb; run.log({'tracking': wandb.Image(f'{cfg.save_dir}/{cfg.run_name}_tracking.png')}); run.finish()

# 03 · Ablation grid
Trains every (variant × head mode × training mode) combination with a shorter budget and reports window metrics,
sequential metrics (own-estimate feedback), parameter count and MACs per inference. Use this to decide:
1. does attention earn its ~60 % of the compute? (`patch` vs `noattn`)
2. do deltas + fusion beat the absolute head on ordered data?
3. does integration-in-the-loop (`K=8`) reduce drift?

In [ ]:
# [session 2] skipped: ablation grid already complete (see ablation_results_merged.csv)
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
# import os, sys, glob, json
# for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
#     if os.path.exists(os.path.join('/kaggle/usr/lib/notebooks/quddusikashaf/emg_common/emg_common.py')):
#         sys.path.insert(0, os.path.abspath(p)); break
# else:
#     raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
# import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
# import emg_common as ec
# pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
# WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
# BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
# EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
# print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# [session 2] skipped: ablation grid already complete (see ablation_results_merged.csv)
# # processed data: output of 01_preprocess_split (same session) or attached as a dataset
# candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
# npz_files = []
# for c in candidates:
#     npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
#     if npz_files: break
# assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
# print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])
# Replace or update the candidate glob logic to point directly to your attached dataset:
# candidates = [
#     f'{WORK}/db2_processed',
#     '/kaggle/input/datasets/quddusikashaf/checkpoint-roformer/db2_processed',
#     '/kaggle/input/checkpoint-roformer/db2_processed'
# ]
# npz_files = []
# for c in candidates:
#     npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
#     if npz_files:
#         break
# assert npz_files, 'no processed data found: check dataset attachment path'
# print(len(npz_files), 'subjects loaded from dataset')

In [ ]:
# [session 2] skipped: ablation grid already complete (see ablation_results_merged.csv)
# base = dict(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{WORK}/outputs_grid', epochs=15, patience=5, qat_epochs=0,
#             use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_ablation')
# GRID = [
    # dict(variant='roformer', heads='absolute', cond_prev=False, unroll_steps=0),   # current design
    # dict(variant='patch',    heads='absolute', cond_prev=False, unroll_steps=0),
    # dict(variant='noattn',   heads='absolute', cond_prev=False, unroll_steps=0),
    # dict(variant='patch',    heads='both',     cond_prev=True,  unroll_steps=0),
    # dict(variant='patch',    heads='both',     cond_prev=True,  unroll_steps=8),
    # dict(variant='noattn',   heads='both',     cond_prev=True,  unroll_steps=8),
#     dict(variant='causal',   heads='both',     cond_prev=True,  unroll_steps=8),
# ]
# if EPOCHS_OVERRIDE is not None:
#     base['epochs'] = EPOCHS_OVERRIDE; GRID = GRID[:int(os.environ.get('EMG_GRID_N', len(GRID)))]
# rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)
# device = ec.get_device(); rows = []

In [ ]:
# [session 2] skipped: ablation grid already complete (see ablation_results_merged.csv)
# for g in GRID:
#     cfg = ec.Config(**base, **g).apply_variant()
#     cfg.run_name = f"{g['variant']}_{g['heads']}_prev{int(g['cond_prev'])}_K{g['unroll_steps']}"
#     run = ec.init_wandb(cfg)
#     model, info = ec.train_model(cfg, npz_files, run=run, tag=cfg.run_name)
#     _, val_loader = ec.make_loaders(ec.Config.from_dict({**cfg.to_dict(), 'unroll_steps': 0}), npz_files)
#     wm = ec.window_metrics(ec.predict_windows(model, val_loader, cfg, device))
#     rep = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr, feedback=True)
#     row = {**g, 'params': ec.count_params(model), 'MACs': ec.estimate_macs(cfg)['total'], 'tokens': cfg.num_tokens,
#            'best_val_loss': min(h['val_loss'] for h in info['history']), **{f'win_{k}': v for k, v in wm.items()},
#            **{f'seq_{k}_rmse': v['rmse'] for k, v in rep.items()}, **{f'seq_{k}_r2': v['r2'] for k, v in rep.items()}}
#     rows.append(row); ec.wandb_log(run, row)
#     if run is not None: run.finish()
#     pd.DataFrame(rows).to_csv(f"{base['save_dir']}/ablation_results.csv", index=False)
# results = pd.DataFrame(rows); display(results)

In [ ]:
# [session 2] skipped: ablation grid already complete (see ablation_results_merged.csv)
# seq_cols = [c for c in results.columns if c.startswith('seq_') and c.endswith('_rmse')]
# best_seq = results[seq_cols].min(axis=1)
# fig, ax = plt.subplots(1, 2, figsize=(16, 5))
# labels = results['variant'] + '/' + results['heads'] + '/K' + results['unroll_steps'].astype(str)
# ax[0].bar(labels, best_seq); ax[0].set_ylabel('best sequential RMSE (normalised angle)'); ax[0].tick_params(axis='x', rotation=30)
# ax[1].scatter(results['MACs'] / 1e6, best_seq)
# for l, x, y in zip(labels, results['MACs'] / 1e6, best_seq): ax[1].annotate(l, (x, y), fontsize=8)
# ax[1].set_xlabel('MACs per inference (M)'); ax[1].set_ylabel('best sequential RMSE'); ax[1].set_xscale('log')
# plt.tight_layout(); plt.savefig(f"{base['save_dir']}/ablation.png", dpi=120); plt.show()
# print('Reference: the current HLS design does ~28 M MACs at ~1 MAC/cycle = ~280 ms at 100 MHz against a 25 ms stride.')

# 04 · Sequential evaluation, drift and fusion
Loads a checkpoint and evaluates it the way it will be deployed: ordered test runs, the model fed its **own** estimate.
Compares estimators: absolute head only, naive delta integration, leaky/clamped integration, Kalman fusion of delta (process) and
absolute (measurement), Kalman + rest re-anchoring, and fixed-point integrators (truncation vs rounding) to show numeric drift.
Also sweeps the Kalman noise parameters and plots error versus time since the run started.

In [ ]:
# # --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
# import os, sys, glob, json
# for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
#     if os.path.exists(os.path.join(p, 'emg_common.py')):
#         sys.path.insert(0, os.path.abspath(p)); break
# else:
#     raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
# import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
# import emg_common as ec
# pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
# WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
# BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
# EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
# print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# # processed data: output of 01_preprocess_split (same session) or attached as a dataset
# candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
# npz_files = []
# for c in candidates:
#     npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
#     if npz_files: break
# assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
# print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
# ---- load the session-2 checkpoint from the attached dataset ----
assert hasattr(ec, 'kalman_sweep'), 're-upload the updated emg_common.py utility script'
ckpts = sorted(glob.glob('/kaggle/input/**/patch_both_prev0_K0_qat5_v2.pt', recursive=True)
               + glob.glob(f'{WORK}/outputs/*_v2.pt'))
CKPT = os.environ.get('EMG_CKPT') or (ckpts[0] if ckpts else None)
assert CKPT, 'attach the session-2 output dataset (outputs/patch_both_prev0_K0_qat5_v2.pt)'
device = ec.get_device()
model, cfg, extra = ec.load_checkpoint(CKPT, device); model.eval()
out_dir = f'{WORK}/seq_eval_v3'; os.makedirs(out_dir, exist_ok=True)
print('checkpoint:', CKPT, '| phase:', extra.get('phase'), '| selected on:', extra.get('select_metric'),
      '| variant:', cfg.variant, cfg.heads, '| cond_prev:', cfg.cond_prev)
rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)

In [ ]:
# ---- sequential evaluation: true-angle seed (optimistic) vs abs-head seed (deployable) ----
reports = {}
for init in ('truth', 'abs'):
    reports[init] = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr,
                                           feedback=True, init=init)
    print(f'--- init={init} ---')
    display(pd.DataFrame({k: {m: v[m] for m in ('rmse', 'r2', 'mae')} for k, v in reports[init].items()}).T.sort_values('rmse').round(4))
summary = pd.DataFrame({init: {k: reports[init][k]['rmse'] for k in ('abs_only', 'abs_smoothed', 'kalman', 'kalman_rest_anchor')}
                        for init in reports}).round(4)
print('RMSE summary (the decisive rows):'); display(summary)
print('delta-head contribution = abs_smoothed - kalman:', (summary.loc['abs_smoothed'] - summary.loc['kalman']).round(4).to_dict())
rep = reports['abs']

In [ ]:
# ---- error versus horizon (deployable seed) ----
hz = pd.DataFrame({k: v['rmse_vs_horizon'] for k, v in rep.items()}); display(hz.round(4))
hz[['abs_only', 'abs_smoothed', 'kalman', 'delta_integrated']].plot(marker='o', figsize=(10, 5), title='RMSE vs steps since run start (init=abs, 1 step = 25 ms)')
plt.ylabel('RMSE'); plt.grid(alpha=.3); plt.savefig(f'{out_dir}/rmse_vs_horizon.png', dpi=120); plt.show()

In [ ]:
# ---- Kalman q/r sweeps: delta fusion vs abs-only smoothing, both with the deployable seed ----
preds = [ec.predict_run(model, r, cfg, device, feedback=True) for f in npz_files for r in ec.ordered_runs(f, 'test') if len(r['angles']) >= 4]
grids = {}
for name, use_delta in (('kalman_with_delta', True), ('abs_smoothing_only', False)):
    grids[name] = ec.kalman_sweep(preds, use_delta=use_delta, init='abs')
    best = grids[name].stack().idxmin(); print(f'{name}: best {best} -> {grids[name].stack().min():.4f}'); display(grids[name].round(4))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, (name, g) in zip(ax, grids.items()):
    im = a.imshow(g.values.astype(float), cmap='viridis'); a.set_title(name); plt.colorbar(im, ax=a)
    a.set_xticks(range(g.shape[1])); a.set_xticklabels(g.columns); a.set_yticks(range(g.shape[0])); a.set_yticklabels(g.index)
plt.tight_layout(); plt.savefig(f'{out_dir}/kalman_sweeps.png', dpi=120); plt.show()
json.dump({'checkpoint': CKPT, 'reports': reports, 'summary': summary.to_dict(),
           'sweeps': {k: g.to_dict() for k, g in grids.items()}}, open(f'{out_dir}/sequential_report_v3.json', 'w'), indent=2)

In [ ]:
# ---- fixed-point integrator drift, deployable seed (why AP_RND and a wide accumulator matter) ----
r = max((r for f in npz_files for r in ec.ordered_runs(f, 'test')), key=lambda x: len(x['angles']))
p = ec.predict_run(model, r, cfg, device, feedback=True); start = p['abs'][0]
j = 3; n = len(r['angles'])
plt.figure(figsize=(14, 5)); plt.plot(r['angles'][:, j], 'k', label='truth'); plt.plot(p['abs'][:, j], 'r', alpha=.4, label='absolute head')
plt.plot(ec.kalman_fuse(np.zeros_like(p['abs']), p['abs'], start)[:, j], 'm', label='abs smoothed')
plt.plot(ec.kalman_fuse(p['delta'], p['abs'], start)[:, j], 'g', lw=2, label='kalman (delta + abs)')
for fb, rd in [(7, 'floor'), (12, 'round')]:
    plt.plot(ec.fixed_point_integrate(p['delta'], start, fb, rd)[:, j], alpha=.6, label=f'integrator {fb} frac bits, {rd}')
plt.title(f'{ec.JOINT_NAMES[j]} over {n} steps (init=abs)'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{out_dir}/fusion_trace.png', dpi=120); plt.show()

# 05 · Export for HLS (one checkpoint → every artifact)
Produces, from a single checkpoint and a **real** test window:
* `verification_data/*.txt` golden intermediates (`conv1d_out`, `l0_rope_out`, `l0_add_norm1_out`, `l0_add_norm2_out`, `pool_out`, `full_output`, …) and float weights `full_*.txt`
* `weights/*.h` quantised headers with the **same array names** the HLS already uses, plus `results.h`, `input_output.h`
* `rope_tables.h`, `config.h` (with new macros: `CONV_STRIDE`, `USE_ATTENTION`, `HAS_DELTA_HEAD`, `COND_PREV`, …), `types_suggested.h` (AP_RND, AP_SAT)
* `hil_verification_samples/` real windows + float outputs + manifest (replaces the inconsistent set in the repo)
* `normalization_constants.{json,h}`: per-channel EMG scale, glove min/max, mu, causal SOS filter, `DELTA_SCALE`
* `fixed_point_expectation.json`: the per-layer error a *correct* HLS build should reproduce

For the unchanged HLS (`variant=roformer`, `heads=absolute`, `cond_prev=False`) the headers are drop-in. Other variants need the HLS changes listed in `config.h` macros.

In [ ]:
# [session 3] skipped: export done in session 2 and verified against the HLS C model
# # --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
# import os, sys, glob, json
# for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
#     if os.path.exists(os.path.join(p, 'emg_common.py')):
#         sys.path.insert(0, os.path.abspath(p)); break
# else:
#     raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
# import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
# import emg_common as ec
# pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
# WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
# BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
# EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
# print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# [session 3] skipped: export done in session 2 and verified against the HLS C model
# # processed data: output of 01_preprocess_split (same session) or attached as a dataset
# candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
# npz_files = []
# for c in candidates:
#     npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
#     if npz_files: break
# assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
# print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
# [session 3] skipped: export done in session 2 and verified against the HLS C model
# checkpoint: the one trained above in this session; fall back to env var / attached datasets
# ckpts = sorted(glob.glob(f'{WORK}/outputs/*_v2.pt')) or sorted(glob.glob(f'{WORK}/outputs/*.pt')) \
#         or sorted(glob.glob('/kaggle/input/*/outputs/*.pt') + glob.glob('/kaggle/input/datasets/*/*/outputs/*.pt'))
# CKPT = os.environ.get('EMG_CKPT') or (info['checkpoint'] if 'info' in globals() else (ckpts[-1] if ckpts else None))
# assert CKPT, 'no checkpoint found'
# device = ec.get_device()
# model, cfg, extra = ec.load_checkpoint(CKPT, device); model.eval()
# cfg.act_fmt, cfg.weight_fmt, cfg.rope_fmt = (10, 3), (10, 3), (8, 2)      # formats to quantise headers/tables with
# cfg.export_weight_type = 'data_t'                                         # 'weight_t' once types.h has the typedef
# EXPORT_DIR = f'{WORK}/hls_export/{os.path.splitext(os.path.basename(CKPT))[0]}'
# print(CKPT, '| variant', cfg.variant, '| heads', cfg.heads, '| cond_prev', cfg.cond_prev, '| tokens', cfg.num_tokens, '| resid_fmt', cfg.resid_fmt)

In [ ]:
# [session 3] skipped: export done in session 2 and verified against the HLS C model
# ---- a real test window for the golden dumps + N real HIL windows spread over the test split ----
# d = np.load(npz_files[0]); test_idx = np.where((d['split'] == 1) & (d['has_prev'] == 1))[0]
# i = int(test_idx[len(test_idx) // 3])
# sample_x = d['sEMG'][i].T.astype(np.float64)                       # [C, seq_len] channel-major = HLS input layout
# sample_prev = d['angles_prev'][i].astype(np.float64) if cfg.cond_prev else None
# N_HIL = int(os.environ.get('EMG_N_HIL', 40))
# hil = [(d['sEMG'][j].T.astype(np.float64), d['angles_prev'][j].astype(np.float64) if cfg.cond_prev else None, int(d['restim'][j]), int(d['rep'][j]))
#        for j in test_idx[np.linspace(0, len(test_idx) - 1, N_HIL).astype(int)]]
# report = ec.export_hls(model, cfg, EXPORT_DIR, sample_x, sample_prev, hil, ec.load_norm_stats(npz_files[0]))
# display(pd.DataFrame(report).T)
# print(open(f'{EXPORT_DIR}/config.h').read()); print(open(f'{EXPORT_DIR}/types_suggested.h').read())

In [ ]:
# [session 3] skipped: export done in session 2 and verified against the HLS C model
# ---- package for download and print the copy instructions ----
# import shutil; zip_path = shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR); print('archive:', zip_path)
# print(f"""
# Copy into the HLS project:
#   {EXPORT_DIR}/weights/*.h            -> hls_code/weights/
#   {EXPORT_DIR}/rope_tables.h          -> hls_code/rope_tables.h
#   {EXPORT_DIR}/config.h               -> hls_code/config.h
#   {EXPORT_DIR}/verification_data/     -> hls_code/verification_data/
#   {EXPORT_DIR}/hil_verification_samples/ -> hil_verification_samples/
#   {EXPORT_DIR}/normalization_constants.h -> vitis_workspace (PS firmware)
# Then re-run C-simulation; per-layer cosine should match fixed_point_expectation.json (about 0.999).
# HLS changes required by this checkpoint: """ + ', '.join(m for m, on in [('strided conv embedding', cfg.conv_stride > 1), ('no attention path', not cfg.use_attention),
#       ('causal mask', cfg.causal), ('RMSNorm', cfg.norm == 'rmsnorm'), ('last-token pooling', cfg.pooling == 'last'),
#       ('delta head output', hasattr(model, 'fc_delta')), ('prev-angle input + prev_w projection', cfg.cond_prev)] if on) or 'none (drop-in)')